# §1.4 ベクトル場と積分曲線 - 多様体上の流れ

## 1. 概要

- **この節で学ぶこと**: ベクトル場の定義、積分曲線（流れ）、リー括弧
- **前提知識**: 接ベクトル、微分方程式の基礎
- **情報幾何との関連**: **勾配流、自然勾配降下法の幾何学的解釈**

## 2. 直感的理解

### ベクトル場とは

- 多様体の各点に接ベクトルを「滑らかに」割り当てたもの
- 流体の速度場、風向き、電場などの一般化

### 積分曲線（フロー）

- ベクトル場に沿って「流される」点の軌跡
- 微分方程式 $\dot{\gamma}(t) = X(\gamma(t))$ の解

### 情報幾何での役割

- **勾配ベクトル場**: $X = -\nabla L$
- **勾配流**: パラメータの最適化軌跡
- **自然勾配流**: Fisher計量を考慮した最適化

### 日常での例え

- ベクトル場 = 川の流れの速度分布
- 積分曲線 = 葉っぱを落としたときの軌跡

## 3. 数学的定義

### 3.1 ベクトル場の定義

多様体 $M$ 上の**ベクトル場** $X$ とは、滑らかな写像
$$X: M \to TM$$
で、各点 $p \in M$ に対して $X(p) \in T_pM$ となるもの。

局所座標 $(x^1, \ldots, x^n)$ では:
$$X = X^i(x) \frac{\partial}{\partial x^i}$$

### 3.2 積分曲線

曲線 $\gamma: I \to M$ がベクトル場 $X$ の**積分曲線**であるとは:
$$\dot{\gamma}(t) = X(\gamma(t))$$

座標成分で:
$$\frac{dx^i}{dt} = X^i(x(t))$$

### 3.3 フロー

$X$ の**フロー** $\phi_t: M \to M$ は:
$$\frac{d}{dt}\phi_t(p) = X(\phi_t(p)), \quad \phi_0(p) = p$$

群の性質:
- $\phi_0 = \text{id}$
- $\phi_s \circ \phi_t = \phi_{s+t}$

### 3.4 リー括弧

2つのベクトル場 $X, Y$ の**リー括弧** $[X, Y]$:
$$[X, Y](f) = X(Y(f)) - Y(X(f))$$

座標成分で:
$$[X, Y]^i = X^j \frac{\partial Y^i}{\partial x^j} - Y^j \frac{\partial X^i}{\partial x^j}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

plt.rcParams['figure.figsize'] = (10, 8)

class VectorField:
    """2次元ベクトル場のクラス"""
    def __init__(self, X_func):
        """
        X_func: (x, y) -> (X1, X2) を返す関数
        """
        self.X_func = X_func
    
    def __call__(self, p):
        """点 p = (x, y) でのベクトル場の値"""
        return np.array(self.X_func(p[0], p[1]))
    
    def integral_curve(self, p0, t_span, n_points=100):
        """点 p0 を出発する積分曲線を計算"""
        t = np.linspace(t_span[0], t_span[1], n_points)
        
        def ode(p, t):
            return self.X_func(p[0], p[1])
        
        curve = odeint(ode, p0, t)
        return t, curve

## 4. 可視化

### 4.1 基本的なベクトル場と積分曲線

In [ ]:
def visualize_basic_vector_fields():
    """Basic vector fields and integral curves"""
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    
    # 4 types of vector fields
    vector_fields = [
        (lambda x, y: (1, 0), 'Translation: X = ∂/∂x'),
        (lambda x, y: (-y, x), 'Rotation: X = -y∂/∂x + x∂/∂y'),
        (lambda x, y: (x, y), 'Expansion: X = x∂/∂x + y∂/∂y'),
        (lambda x, y: (-x, -2*y), 'Contraction (anisotropic): X = -x∂/∂x - 2y∂/∂y')
    ]
    
    for ax, (X_func, title) in zip(axes.flatten(), vector_fields):
        X = VectorField(X_func)
        
        # Draw vector field as arrows
        x = np.linspace(-2, 2, 15)
        y = np.linspace(-2, 2, 15)
        XX, YY = np.meshgrid(x, y)
        
        U = np.zeros_like(XX)
        V = np.zeros_like(YY)
        for i in range(len(x)):
            for j in range(len(y)):
                vec = X([XX[j, i], YY[j, i]])
                U[j, i] = vec[0]
                V[j, i] = vec[1]
        
        ax.quiver(XX, YY, U, V, color='blue', alpha=0.6)
        
        # Integral curves
        for p0 in [[-1.5, 0], [0, 1.5], [1, 1], [-1, -1]]:
            try:
                t, curve = X.integral_curve(p0, [0, 3], n_points=100)
                ax.plot(curve[:, 0], curve[:, 1], 'r-', linewidth=2, alpha=0.7)
                ax.plot(p0[0], p0[1], 'go', markersize=8)
            except:
                pass
        
        ax.set_xlim(-2.5, 2.5)
        ax.set_ylim(-2.5, 2.5)
        ax.set_aspect('equal')
        ax.set_title(title)
        ax.grid(True, alpha=0.3)
        ax.axhline(0, color='gray', linewidth=0.5)
        ax.axvline(0, color='gray', linewidth=0.5)
    
    plt.tight_layout()
    plt.show()

visualize_basic_vector_fields()

### 4.2 勾配ベクトル場（最適化への接続）

In [ ]:
def visualize_gradient_flow():
    """Gradient vector field and gradient flow"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Loss function L(x, y) = x² + 4y² (elliptical contours)
    def L(x, y):
        return x**2 + 4*y**2
    
    def grad_L(x, y):
        return np.array([2*x, 8*y])
    
    x = np.linspace(-2, 2, 50)
    y = np.linspace(-2, 2, 50)
    X, Y = np.meshgrid(x, y)
    Z = L(X, Y)
    
    # Left: Loss function contours and negative gradient field
    ax1 = axes[0]
    
    cs = ax1.contour(X, Y, Z, levels=[0.5, 1, 2, 4, 6], colors='blue', alpha=0.5)
    ax1.clabel(cs, inline=True, fontsize=10)
    
    # Negative gradient vector field
    skip = 5
    ax1.quiver(X[::skip, ::skip], Y[::skip, ::skip],
               -2*X[::skip, ::skip], -8*Y[::skip, ::skip],
               color='red', alpha=0.5, scale=40)
    
    # Gradient flow (integral curves)
    neg_grad = VectorField(lambda x, y: (-2*x, -8*y))
    
    for p0 in [[1.5, 0.8], [-1.8, 0.5], [0.5, -1.0], [-1, -0.8]]:
        t, curve = neg_grad.integral_curve(p0, [0, 2], n_points=100)
        ax1.plot(curve[:, 0], curve[:, 1], 'g-', linewidth=2)
        ax1.plot(p0[0], p0[1], 'go', markersize=10)
    
    ax1.plot(0, 0, 'r*', markersize=20, label='Minimum')
    ax1.set_xlabel('x')
    ax1.set_ylabel('y')
    ax1.set_title('Gradient Descent\n$\\dot{\\theta} = -\\nabla L$')
    ax1.set_aspect('equal')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Right: Natural gradient flow with Fisher metric
    ax2 = axes[1]
    
    # Assume Fisher metric g = diag(1, 4)
    # Natural gradient = g^{-1} ∇L
    def natural_grad_L(x, y):
        grad = grad_L(x, y)
        g_inv = np.diag([1, 0.25])  # g^{-1} = diag(1, 1/4)
        return g_inv @ grad
    
    cs = ax2.contour(X, Y, Z, levels=[0.5, 1, 2, 4, 6], colors='blue', alpha=0.5)
    ax2.clabel(cs, inline=True, fontsize=10)
    
    # Negative natural gradient vector field
    U_nat = -2*X  # g^{-1} (1,1) component × ∂L/∂x
    V_nat = -2*Y  # g^{-1} (2,2) component × ∂L/∂y = 0.25 × 8y = 2y
    ax2.quiver(X[::skip, ::skip], Y[::skip, ::skip],
               U_nat[::skip, ::skip], V_nat[::skip, ::skip],
               color='purple', alpha=0.5, scale=30)
    
    # Natural gradient flow
    nat_grad = VectorField(lambda x, y: (-2*x, -2*y))
    
    for p0 in [[1.5, 0.8], [-1.8, 0.5], [0.5, -1.0], [-1, -0.8]]:
        t, curve = nat_grad.integral_curve(p0, [0, 2], n_points=100)
        ax2.plot(curve[:, 0], curve[:, 1], 'orange', linewidth=2)
        ax2.plot(p0[0], p0[1], 'o', color='orange', markersize=10)
    
    ax2.plot(0, 0, 'r*', markersize=20, label='Minimum')
    ax2.set_xlabel('x')
    ax2.set_ylabel('y')
    ax2.set_title('Natural Gradient Descent\n$\\dot{\\theta} = -I(\\theta)^{-1}\\nabla L$')
    ax2.set_aspect('equal')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("[Observation]")
    print("- Gradient descent (left): Converges fast in y-direction (larger gradient)")
    print("- Natural gradient (right): Isotropic convergence (corrected by Fisher metric)")

visualize_gradient_flow()

### 4.3 リー括弧の幾何学的意味

In [ ]:
def visualize_lie_bracket():
    """Geometric interpretation of Lie bracket"""
    fig, ax = plt.subplots(figsize=(10, 10))
    
    # Two vector fields
    # X: flow in x-direction
    # Y: y-direction flow proportional to x
    X = VectorField(lambda x, y: (1, 0))
    Y = VectorField(lambda x, y: (0, x))
    
    # Calculation of [X, Y]
    # [X, Y]^1 = X^j ∂Y^1/∂x^j - Y^j ∂X^1/∂x^j = 1·0 - x·0 = 0
    # [X, Y]^2 = X^j ∂Y^2/∂x^j - Y^j ∂X^2/∂x^j = 1·1 - x·0 = 1
    # Therefore [X, Y] = ∂/∂y
    
    # Starting point
    p0 = np.array([0, 0])
    eps = 0.5
    
    # Move eps in X direction
    p1 = p0 + eps * X(p0)  # (0.5, 0)
    
    # Move eps in Y direction
    p2 = p1 + eps * Y(p1)  # (0.5, 0.25)
    
    # Move eps in -X direction
    p3 = p2 + eps * (-X(p2))  # (0, 0.25)
    
    # Move eps in -Y direction
    p4 = p3 + eps * (-Y(p3))  # (0, 0.25) - (0, 0) = (0, 0.25)
    # Note: Y(p3) = (0, 0) so it doesn't return!
    
    # Exact calculation
    print(f"Starting point p0 = {p0}")
    print(f"X(p0) = {X(p0)}, p1 = p0 + eps·X(p0) = {p1}")
    print(f"Y(p1) = {Y(p1)}, p2 = p1 + eps·Y(p1) = {p2}")
    print(f"-X(p2) = {-X(p2)}, p3 = p2 - eps·X(p2) = {p3}")
    print(f"-Y(p3) = {-Y(p3)}, p4 = p3 - eps·Y(p3) = {p4}")
    print(f"")
    print(f"Does not close! Gap = p4 - p0 = {p4 - p0}")
    print(f"Gap ≈ eps^2 [X,Y](p0) = eps^2 d/dy|_p0 = (0, {eps**2})")
    
    # Visualization
    # Draw vector fields
    x = np.linspace(-0.5, 1.5, 10)
    y = np.linspace(-0.5, 1, 10)
    XX, YY = np.meshgrid(x, y)
    
    # X vector field
    ax.quiver(XX, YY, np.ones_like(XX), np.zeros_like(YY), 
              color='blue', alpha=0.3, scale=15, label='X = d/dx')
    
    # Y vector field
    ax.quiver(XX, YY, np.zeros_like(XX), XX, 
              color='red', alpha=0.3, scale=15, label='Y = x d/dy')
    
    # Draw rectangular path
    path_x = [p0[0], p1[0], p2[0], p3[0], p4[0]]
    path_y = [p0[1], p1[1], p2[1], p3[1], p4[1]]
    
    ax.plot([p0[0], p1[0]], [p0[1], p1[1]], 'b-', linewidth=3, label='+X direction')
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'r-', linewidth=3, label='+Y direction')
    ax.plot([p2[0], p3[0]], [p2[1], p3[1]], 'b--', linewidth=3, label='-X direction')
    ax.plot([p3[0], p4[0]], [p3[1], p4[1]], 'r--', linewidth=3, label='-Y direction')
    
    # Highlight the gap
    ax.annotate('', xy=p4, xytext=p0, 
                arrowprops=dict(arrowstyle='->', color='green', lw=3))
    ax.text(0.05, 0.12, f'Gap ≈ eps^2[X,Y]', fontsize=12, color='green')
    
    # Mark points
    for i, (px, py, label) in enumerate([(p0[0], p0[1], 'p0'), (p1[0], p1[1], 'p1'),
                                          (p2[0], p2[1], 'p2'), (p3[0], p3[1], 'p3'),
                                          (p4[0], p4[1], 'p4')]):
        ax.plot(px, py, 'ko', markersize=10)
        ax.annotate(label, (px, py), xytext=(5, 5), textcoords='offset points', fontsize=12)
    
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1)
    ax.set_aspect('equal')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title('Geometric Meaning of Lie Bracket\nX->Y->(-X)->(-Y) does not close -> [X,Y] != 0')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    
    plt.show()

visualize_lie_bracket()

### 4.4 位相空間でのベクトル場（力学系）

In [ ]:
def visualize_phase_space():
    """Vector field in phase space (pendulum example)"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Damped pendulum: theta'' + gamma*theta' + sin(theta) = 0
    # Phase space: (theta, omega) where omega = theta'
    # theta' = omega
    # omega' = -sin(theta) - gamma*omega
    
    gamma = 0.3  # Damping coefficient
    
    def pendulum_field(theta, omega):
        return (omega, -np.sin(theta) - gamma * omega)
    
    X = VectorField(lambda x, y: pendulum_field(x, y))
    
    # Left: Vector field
    ax1 = axes[0]
    
    theta = np.linspace(-np.pi, np.pi, 20)
    omega = np.linspace(-2, 2, 20)
    THETA, OMEGA = np.meshgrid(theta, omega)
    
    U = OMEGA
    V = -np.sin(THETA) - gamma * OMEGA
    
    # Color by speed
    speed = np.sqrt(U**2 + V**2)
    ax1.streamplot(THETA, OMEGA, U, V, color=speed, cmap='coolwarm', density=1.5)
    
    ax1.plot(0, 0, 'go', markersize=15, label='Stable equilibrium')
    ax1.plot([-np.pi, np.pi], [0, 0], 'ro', markersize=15, label='Unstable equilibrium')
    
    ax1.set_xlabel(r'$\theta$')
    ax1.set_ylabel(r'$\omega = \dot{\theta}$')
    ax1.set_title(f'Phase Space of Damped Pendulum\ngamma = {gamma}')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Right: Several trajectories
    ax2 = axes[1]
    
    # Initial conditions
    initial_conditions = [
        (0.5, 0), (1.0, 0), (2.0, 0), (2.5, 0),
        (-0.5, 1), (0, 1.5), (0, -1.5)
    ]
    
    colors = plt.cm.viridis(np.linspace(0, 1, len(initial_conditions)))
    
    for (theta0, omega0), color in zip(initial_conditions, colors):
        t, curve = X.integral_curve([theta0, omega0], [0, 20], n_points=500)
        ax2.plot(curve[:, 0], curve[:, 1], color=color, linewidth=1.5, alpha=0.8)
        ax2.plot(theta0, omega0, 'o', color=color, markersize=8)
    
    ax2.plot(0, 0, 'g*', markersize=20, label='Attractor')
    
    ax2.set_xlabel(r'$\theta$')
    ax2.set_ylabel(r'$\omega$')
    ax2.set_title('Trajectories in Phase Space\n(All converge to origin)')
    ax2.set_xlim(-3, 3)
    ax2.set_ylim(-2.5, 2.5)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

visualize_phase_space()

## 5. 具体例

### 例1：リー括弧の計算

In [ ]:
def compute_lie_bracket():
    """リー括弧の計算例"""
    print("【リー括弧の計算】")
    print()
    print("X = x∂/∂x + y∂/∂y  (拡大)")
    print("Y = -y∂/∂x + x∂/∂y  (回転)")
    print()
    print("[X, Y]^i = X^j ∂Y^i/∂x^j - Y^j ∂X^i/∂x^j")
    print()
    
    # X = (x, y), Y = (-y, x)
    # ∂Y^1/∂x = 0, ∂Y^1/∂y = -1
    # ∂Y^2/∂x = 1, ∂Y^2/∂y = 0
    # ∂X^1/∂x = 1, ∂X^1/∂y = 0
    # ∂X^2/∂x = 0, ∂X^2/∂y = 1
    
    print("[X, Y]^1 = x·0 + y·(-1) - (-y)·1 - x·0 = -y + y = 0")
    print("[X, Y]^2 = x·1 + y·0 - (-y)·0 - x·1 = x - x = 0")
    print()
    print("したがって [X, Y] = 0")
    print()
    print("【幾何学的意味】")
    print("拡大と回転は『可換』= どちらを先に行っても同じ結果")

compute_lie_bracket()

### 例2：勾配流の解析解

In [ ]:
def analyze_gradient_flow():
    """2次関数の勾配流の解析解"""
    print("【勾配流の解析解】")
    print()
    print("損失関数: L(x, y) = (1/2)(ax² + by²)")
    print("勾配: ∇L = (ax, by)")
    print("勾配流: dx/dt = -ax, dy/dt = -by")
    print()
    print("解: x(t) = x₀ e^{-at}, y(t) = y₀ e^{-bt}")
    print()
    
    # 具体例
    a, b = 1, 4
    x0, y0 = 2, 1
    
    print(f"a = {a}, b = {b} の場合:")
    print(f"  x(t) = {x0} e^{{-{a}t}}")
    print(f"  y(t) = {y0} e^{{-{b}t}}")
    print()
    print("y方向（b=4）の方が速く収束")
    print("これは『条件数』による収束速度の違い")
    print()
    
    # 自然勾配の場合
    print("【自然勾配（Fisher計量 g = diag(a, b) の場合）】")
    print("自然勾配: g⁻¹∇L = (x, y)")
    print("自然勾配流: dx/dt = -x, dy/dt = -y")
    print("解: x(t) = x₀ e^{-t}, y(t) = y₀ e^{-t}")
    print()
    print("→ 両方向で同じ速度で収束！")

analyze_gradient_flow()

### 例3：発散（ダイバージェンス）

In [ ]:
def demonstrate_divergence():
    """Divergence of vector fields"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    fields = [
        (lambda x, y: (x, y), 'X = (x, y)\ndiv X = 2 > 0 (source)'),
        (lambda x, y: (-y, x), 'X = (-y, x)\ndiv X = 0 (incompressible)'),
        (lambda x, y: (-x, -y), 'X = (-x, -y)\ndiv X = -2 < 0 (sink)')
    ]
    
    for ax, (X_func, title) in zip(axes, fields):
        x = np.linspace(-2, 2, 15)
        y = np.linspace(-2, 2, 15)
        X, Y = np.meshgrid(x, y)
        
        U, V = X_func(X, Y)
        
        ax.quiver(X, Y, U, V, color='blue', alpha=0.7)
        ax.plot(0, 0, 'ro', markersize=10)
        
        ax.set_xlim(-2.5, 2.5)
        ax.set_ylim(-2.5, 2.5)
        ax.set_aspect('equal')
        ax.set_title(title)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("[Divergence Calculation]")
    print("div X = dX^1/dx + dX^2/dy")
    print()
    print("Relevance to Information Geometry:")
    print("- Flow of probability density (continuity equation)")
    print("- Fokker-Planck equation")

demonstrate_divergence()

## 6. 他の概念との関係

### 前の節との繋がり
- **接ベクトル (§1.2)**: ベクトル場は各点での接ベクトルの集まり
- **余接ベクトル (§1.3)**: 1-形式場は各点での余接ベクトルの集まり

### 次の節への接続
- **リーマン計量 (§1.5)**: 勾配ベクトル場の定義に計量が必要
- **アファイン接続 (§1.6)**: ベクトル場の微分（共変微分）

### 情報幾何との関連

| 概念 | 一般の微分幾何 | 情報幾何 |
|:---|:---|:---|
| ベクトル場 | $X = X^i \partial_i$ | 統計多様体上の方向 |
| 勾配場 | $\nabla f$ | 損失関数の勾配 |
| フロー | $\phi_t$ | 最適化の軌跡 |
| 自然勾配 | $g^{-1} \nabla f$ | Fisher計量での勾配 |

### 重要な関係式

- **フローと微分**: $\frac{d}{dt}(f \circ \phi_t) = X(f) \circ \phi_t$
- **リー微分**: $\mathcal{L}_X Y = [X, Y]$
- **勾配流の収束**: $\frac{d}{dt}L(\theta(t)) = -\|\nabla L\|^2 \leq 0$

## 7. 演習問題

### Q1. 積分曲線

ベクトル場 $X = y\frac{\partial}{\partial x} - x\frac{\partial}{\partial y}$ の、点 $(1, 0)$ を通る積分曲線を求めよ。

<details>
<summary>解答を見る</summary>

微分方程式:
$$\frac{dx}{dt} = y, \quad \frac{dy}{dt} = -x$$

解: $x(t) = \cos t$, $y(t) = -\sin t$（反時計回りの円）

（$x^2 + y^2 = 1$ を保存）

</details>

In [ ]:
# Q1 Verification
X = VectorField(lambda x, y: (y, -x))
t, curve = X.integral_curve([1, 0], [0, 2*np.pi], n_points=100)

plt.figure(figsize=(6, 6))
plt.plot(curve[:, 0], curve[:, 1], 'b-', linewidth=2)
plt.plot(1, 0, 'go', markersize=10, label='Initial point (1, 0)')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Integral Curve (Unit Circle)')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### Q2. リー括弧

$X = \frac{\partial}{\partial x}$, $Y = x\frac{\partial}{\partial y}$ のリー括弧 $[X, Y]$ を計算せよ。

<details>
<summary>解答を見る</summary>

$X = (1, 0)$, $Y = (0, x)$

$[X, Y]^1 = 1 \cdot 0 + 0 \cdot 0 - 0 \cdot 0 - x \cdot 0 = 0$

$[X, Y]^2 = 1 \cdot 1 + 0 \cdot 0 - 0 \cdot 0 - x \cdot 0 = 1$

よって $[X, Y] = \frac{\partial}{\partial y}$

</details>

### Q3. 勾配流の収束

$L(x, y) = x^2 + y^2$ の勾配流 $\dot{\theta} = -\nabla L$ について、$L(\theta(t))$ の時間発展を求めよ。

<details>
<summary>解答を見る</summary>

$\nabla L = (2x, 2y)$ なので、$\dot{x} = -2x$, $\dot{y} = -2y$

解: $x(t) = x_0 e^{-2t}$, $y(t) = y_0 e^{-2t}$

よって: $L(\theta(t)) = x(t)^2 + y(t)^2 = (x_0^2 + y_0^2) e^{-4t} = L_0 e^{-4t}$

指数関数的に0に収束する。

</details>

In [ ]:
# Q3 Verification
def L(x, y):
    return x**2 + y**2

X = VectorField(lambda x, y: (-2*x, -2*y))
t, curve = X.integral_curve([2, 1], [0, 3], n_points=100)

L_t = [L(c[0], c[1]) for c in curve]
L0 = L(2, 1)
L_analytical = L0 * np.exp(-4*t)

plt.figure(figsize=(8, 5))
plt.plot(t, L_t, 'b-', linewidth=2, label='Numerical solution')
plt.plot(t, L_analytical, 'r--', linewidth=2, label=r'Analytical: $L_0 e^{-4t}$')
plt.xlabel('Time t')
plt.ylabel('L(theta(t))')
plt.title('Loss Function Decay via Gradient Flow')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. 参考：使用したプロンプト

```
ベクトル場と積分曲線の関係を、流体の速度場と粒子の軌跡の
アナロジーで説明してください。
```

```
勾配降下法と自然勾配降下法の違いを、ベクトル場の観点から
可視化するPythonコードを書いてください。同じ損失関数でも
流れが違うことを示してください。
```

```
リー括弧 [X, Y] の幾何学的意味を、「X→Y→(-X)→(-Y)で閉じない」
という観点から説明してください。具体例で図示してください。
```

```
減衰振り子の位相空間での軌道を描くPythonコードを書いてください。
ベクトル場のストリームプロットも表示してください。
```

---
**次のステップ**: `ch01_differential_geometry/05_riemannian_metric.ipynb` へ